# Embedding Model Benchmark Lab
### Interactive Exploration, Representation Learning, and Retrieval Evaluation

Welcome to the **Embedding Model Benchmark Lab** exploration notebook! This notebook provides an interactive walkthrough of:
1. **Dense Vector Representations**: How text is mapped to points in high-dimensional vector spaces.
2. **The BEIR NFCorpus Dataset**: Real-world medical semantic search data with human relevance judgments.
3. **Similarity Search**: Computing cosine similarity using dot products of normalized vectors.
4. **Information Retrieval Metrics**: Step-by-step calculation of Recall@K, MRR@K, and nDCG@K.

## 1. Imports and Configuration Setup

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import yaml
import numpy as np
import pandas as pd
from src.data.dataset_loader import DatasetLoader
from src.models.loader import EmbeddingModelWrapper
from src.evaluation.metrics import recall_at_k, precision_at_k, mrr_at_k, ndcg_at_k

print("✓ Core modules successfully imported!")

## 2. Ingest and Inspect the NFCorpus Dataset
Let's load the BEIR NFCorpus dataset and inspect real medical queries and documents.

In [ ]:
loader = DatasetLoader(data_dir=str(project_root / "data" / "nfcorpus"))
dataset = loader.load_dataset(split="test", auto_download=True)

# Sample inspection
sample_qid = list(dataset.queries.keys())[0]
sample_query = dataset.queries[sample_qid]
sample_qrels = dataset.qrels[sample_qid]

print(f"Sample Query ID: {sample_qid}")
print(f"Query Text: '{sample_query}'")
print(f"Number of Gold Relevant Documents: {len(sample_qrels)}")
for did, score in list(sample_qrels.items())[:3]:
    doc = dataset.corpus.get(did, {'title': 'N/A', 'text': 'N/A'})
    print(f"  → Doc [{did}] (Relevance {score}): {doc['title'][:60]}...")

## 3. Generate Dense Embeddings & Compute Cosine Similarity
We load `all-MiniLM-L6-v2` and project both the query and candidate documents into 384-dimensional space.

In [ ]:
wrapper = EmbeddingModelWrapper(
    model_id="sentence-transformers/all-MiniLM-L6-v2",
    name="all-MiniLM-L6-v2",
    device="cpu"
)
profile = wrapper.load()
print(f"Model Architecture: {profile.architecture}")
print(f"Dimension: {profile.dimension} | Parameter Count: {profile.parameter_count / 1e6:.1f}M")

# Pick 5 candidate documents (including relevant ones)
candidate_doc_ids = list(sample_qrels.keys())[:3] + [did for did in list(dataset.corpus.keys()) if did not in sample_qrels][:3]
candidate_texts = [f"{dataset.corpus[did]['title']} {dataset.corpus[did]['text']}" for did in candidate_doc_ids]

# Encode query and candidate documents
query_vec = wrapper.encode_queries([sample_query], normalize_embeddings=True)[0]
candidate_vecs = wrapper.encode_corpus(candidate_texts, normalize_embeddings=True)

# Cosine similarities via inner product
similarities = np.dot(candidate_vecs, query_vec)
ranked_indices = np.argsort(-similarities)

print(f"\nSearch Results for: '{sample_query}'")
print("-" * 80)
for rank, idx in enumerate(ranked_indices, start=1):
    did = candidate_doc_ids[idx]
    sim = similarities[idx]
    gold_rel = sample_qrels.get(did, 0)
    print(f"Rank {rank}: Score={sim:.4f} | Gold Rel={gold_rel} | Doc ID={did} | Title: {dataset.corpus[did]['title'][:40]}")

## 4. Understanding Ranking Metrics: Step-by-Step
- **Recall@K**: Did our Top-K contain the relevant items?
- **MRR@K**: What was the reciprocal rank (1 / rank) of the very first relevant hit?
- **nDCG@K**: Did highly relevant items appear at the top, discounted logarithmically by their position?

In [ ]:
retrieved_ids = [candidate_doc_ids[idx] for idx in ranked_indices]
relevant_set = {did for did, score in sample_qrels.items() if score > 0}

print(f"Recall@1:    {recall_at_k(retrieved_ids, relevant_set, k=1):.4f}")
print(f"Recall@3:    {recall_at_k(retrieved_ids, relevant_set, k=3):.4f}")
print(f"Precision@3: {precision_at_k(retrieved_ids, relevant_set, k=3):.4f}")
print(f"MRR@3:       {mrr_at_k(retrieved_ids, relevant_set, k=3):.4f}")
print(f"nDCG@3:      {ndcg_at_k(retrieved_ids, sample_qrels, k=3):.4f}")

## 5. Visualizing Benchmark Results
Inspect the full automated benchmark results across models.

In [ ]:
summary_csv = project_root / "results" / "tables" / "benchmark_summary_latest.csv"
if summary_csv.exists():
    df_summary = pd.read_csv(summary_csv)
    print("Benchmark Results Table:")
    display(df_summary)
else:
    print("Run 'python scripts/run_benchmark.py' to generate benchmark results summary.")